In [1]:
from datasets import load_dataset

ds = load_dataset("gmanolache/CrypticBio", split="train")

Resolving data files:   0%|          | 0/631 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/180 [00:00<?, ?it/s]

In [2]:
print(ds)
ds[0]

Dataset({
    features: ['scientificName', 'year', 'month', 'day', 'decimalLatitude', 'decimalLongitude', 'url', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'vernacularName', 'crypticGroup'],
    num_rows: 171832985
})


{'scientificName': 'Salticus scenicus',
 'year': 2022.0,
 'month': 3.0,
 'day': 16.0,
 'decimalLatitude': 49.682217,
 'decimalLongitude': -124.975088,
 'url': 'https://inaturalist-open-data.s3.amazonaws.com/photos/183187792/original.jpg',
 'kingdom': 'Animalia',
 'phylum': 'Arthropoda',
 'class': 'Arachnida',
 'order': 'Araneae',
 'family': 'Salticidae',
 'genus': 'Salticus',
 'vernacularName': ['Zebra Jumper',
  'Zebra Spider',
  'Zebra Jumping Spider',
  'Zebra Back Spider'],
 'crypticGroup': ['Phidippus audax',
  'Salticus cingulatus',
  'Paraphidippus aurantius',
  'Platycryptus undatus',
  'Attulus fasciger',
  'Naphrys pulex',
  'Marpissa muscosa',
  'Eris militaris',
  'Menemerus semilimbatus',
  'Menemerus bivittatus',
  'Hasarius adansoni',
  'Phidippus putnami',
  'Pelegrina galathea',
  'Maevia inclemens',
  'Pseudeuophrys lanigera',
  'Phidippus otiosus']}

# We need to filter to a smaller sub-set of data so that experimetns are more managable 

In [3]:

#Load just the benchmark files
common =load_dataset("gmanolache/CrypticBio", data_files="CrypticBio-Benchmark/CrypticBio-Common.csv", split="train")
print(common.features)

{'scientificName': Value('string'), 'year': Value('float64'), 'month': Value('float64'), 'day': Value('float64'), 'decimalLatitude': Value('float64'), 'decimalLongitude': Value('float64'), 'url': Value('string'), 'kingdom': Value('string'), 'phylum': Value('string'), 'class': Value('string'), 'order': Value('string'), 'family': Value('string'), 'genus': Value('string'), 'vernacularName': Value('string')}


In [4]:
#get unique pairs from the benchmark
unique_names = set(common["scientificName"])

#batch filtering to get the common subset
common_subset = ds.filter(
    lambda x: [
        name in unique_names 
        for name in x['scientificName']
    ],
    batched=True,
    batch_size=10000
)


In [5]:
import requests
from PIL import Image
from io import BytesIO
import torch
import numpy as np


# Open the image from the URL
def open_image(url : list[str]):
    try:
        response = requests.get(url, timeout=10)
        return Image.open(BytesIO(response.content))
    except requests.exceptions.RequestException as e:
        print(f"Error downloading image: {e}")
        return None

# Example usage
img_url = common[12985]['url']
image = open_image(img_url)
image.show()

In [6]:
import open_clip

# Load the model and preprocessors
model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms('hf-hub:imageomics/bioclip-2')
tokenizer = open_clip.get_tokenizer('hf-hub:imageomics/bioclip-2')


#tojenize the text from family and genus - same as cryptic group names
def tokenize_family_genius(sample: dict):
    family_name = sample['family']
    genus_name = sample['genus']
    return tokenizer(family_name + ' ' + genus_name)

#preprocess the image and add batch dimension  
def preprocess_image(sample: dict):
    img_url = sample['url']
    image = open_image(img_url)
    if image is not None:
        return preprocess_val(image).unsqueeze(0)  #add batch dimension
    else:
        return None






In [7]:
#take a random list of uniqeue familises 
list_of_uniqe_families = set(common_subset['scientificName'][np.random.choice(len(common_subset), size=200, replace=False)]) 
list_of_uniqe_families = list(list_of_uniqe_families)[:9] 


## Experiment 1 - adding Date (month + day) to the existent embeddings 

Encoding the date(month,day), using sine and cosine following the intuition from: https://harrisonpim.com/blog/the-best-way-to-encode-dates-times-and-other-cyclical-features . We kinda mimic the first stage of the BioClip training - learing meaningful represemntation of date

In [8]:
def encode_date(month, day):
    """
    Encode month and day as cyclical features using sin/cos projections.
    Each cyclical component maps to a point on a unit circle,
    so December (12) is close to January (1), and day 365 is close to day 1.
    """
    # Normalize to [0, 1] range
    month_norm = (month - 1) / 12       
    day_of_year_norm = ((month - 1) * 30 + day) / 365 

    # Project onto unit circle using sin/cos
    month_sin = (np.sin(2 * np.pi * month_norm) + 1) / 2
    month_cos = (np.cos(2 * np.pi * month_norm) + 1) / 2
    day_sin = (np.sin(2 * np.pi * day_of_year_norm) + 1) / 2
    day_cos = (np.cos(2 * np.pi * day_of_year_norm) + 1) / 2

    return torch.tensor([month_sin, month_cos, day_sin, day_cos], dtype=torch.float32)

In [9]:
example = common_subset[0]
date_features = encode_date(example['month'], example['day'])
print(date_features)

tensor([0.5000, 0.0000, 0.3682, 0.0177])


Create a NN that maps dates to species

In [10]:
import torch.nn as nn

# This network is prett naive and prob can be improved

class Date_to_species_Common(nn.Module):
    """Maps 4-dim date features into one of Cryptic Bio species Scientific Name"""
    def __init__(self, date_dim=4, output_dim=158): # 158 unique species in the common subset
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(date_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 512),
            nn.ReLU(),
            nn.Linear(512, output_dim),
                
        )

    def forward(self, x):
       x=torch.flatten(x, start_dim=1)  # Flatten the input to (batch_size, date_dim)
       return self.network(x)
        

** ** 

In [12]:
from tqdm import tqdm #to monitor progress
BATCH = 256
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
scientific_names = common_subset['scientificName']

unique_names = sorted(set(scientific_names))

name_to_idx = {name: i for i, name in enumerate(unique_names)}

# Encode using BioCLI in batches
species_text_embs = []
with torch.no_grad(): 
    for i in tqdm(range(0, len(unique_names), BATCH), desc="Encoding names"):
        batch = unique_names[i:i+BATCH]
        tokens = tokenizer(batch).to(device)
        #normalize the embeddings to unit length to match CLIP space
        embs = model.encode_text(tokens)
        embs /= embs.norm(dim=-1, keepdim=True)
        species_text_embs.append(embs.cpu()) #move to CPU to save GPU memory

# Concatenate all batches into a single tensor
species_text_embs = torch.cat(species_text_embs, dim=0)


Encoding names: 100%|██████████| 1/1 [00:00<00:00,  1.46it/s]


# Train separate weak model on mapping dates to species, to give only a rough estimate

Add training loop for the date to species network and evaluate on the test set

In [13]:
from torch.utils.data import DataLoader

common_subset = common_subset.filter(lambda x: x['month'] is not None and x['day'] is not None and x['decimalLatitude'] is not None and x['decimalLongitude'] is not None)

processed = common_subset.map(lambda x: {
    'encoded_input': encode_date(x['month'], x['day']),
    'label_idx': name_to_idx[x['scientificName']],
    'coords': torch.tensor([x['decimalLatitude'], x['decimalLongitude']], dtype=torch.float32)
})
#processed = processed.shuffle(seed=42).select(range(2_000_000) ) #take a random subset for training to speed up
processed.set_format(type='torch', columns=['encoded_input', 'label_idx', 'url', 'coords']) #set format for PyTorch DataLoader

train_ds, test_ds = processed.train_test_split(test_size=0.21, seed=42).values()
train_loader = DataLoader(train_ds, batch_size=2048, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=2048, shuffle=False, num_workers=2, pin_memory=True)




Filter:   0%|          | 0/6497834 [00:00<?, ? examples/s]

Map:   0%|          | 0/6495509 [00:00<?, ? examples/s]

In [14]:
processed.set_format(type='torch', columns=['encoded_input', 'label_idx', 'url', 'coords']) #set format for PyTorch DataLoader

train_ds, test_ds = processed.train_test_split(test_size=0.21, seed=42).values()
train_loader = DataLoader(train_ds, batch_size=2048, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=2048, shuffle=False, num_workers=2, pin_memory=True)


In [15]:
print(processed[0])

{'url': 'https://inaturalist-open-data.s3.amazonaws.com/photos/995027/original.JPG', 'encoded_input': tensor([0.5000, 0.0000, 0.3682, 0.0177]), 'label_idx': tensor(28), 'coords': tensor([  37.3627, -122.1761])}


In [ ]:
network_date_to_species = Date_to_species_Common().to(device)
loss=nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(network_date_to_species.parameters(), lr=1e-3)
for epoch in range(10):
    total_loss, steps = 0.0, 0
    for batch in train_loader:
        x = batch['encoded_input'].to(device, non_blocking=True)
        y = batch['label_idx'].to(device, non_blocking=True)
        optimizer.zero_grad()
        logits = network_date_to_species(x)
        l = loss(logits, y)
        l.backward()
        optimizer.step()
        total_loss += l.item(); steps += 1
    print(f"Epoch {epoch} | train loss: {total_loss/steps:.4f}")

Epoch 0 | train loss: 3.8943
Epoch 1 | train loss: 3.8804
Epoch 2 | train loss: 3.8788
Epoch 3 | train loss: 3.8779
Epoch 4 | train loss: 3.8772
Epoch 5 | train loss: 3.8766
Epoch 6 | train loss: 3.8761
Epoch 7 | train loss: 3.8757
Epoch 8 | train loss: 3.8754
Epoch 9 | train loss: 3.8751


In [17]:
labels = processed[:]['label_idx']  
counts = torch.bincount(labels, minlength=len(name_to_idx)).float()
p = counts / counts.sum()
marginal_loss = -(p * torch.log(p + 1e-12)).sum().item()
print(marginal_loss)

4.083083629608154


Use the test set to confirm any improvement

In [ ]:
test_labels = []
test_images = []
preprocessed_images = []
preprocessed_dates = []
coords = []
target = 1000
failed = 0

for batch in test_loader:
    batch_size = len(batch['url'])
    for i in range(batch_size):
        if len(preprocessed_images) >= target:
            break

        url = batch['url'][i]
        try:
            img = preprocess_image({'url': url})
            if img is None:
                failed += 1
                continue

            # Only append after the image succeeds 
            preprocessed_images.append(img)
            test_images.append(url)
            test_labels.append(batch['label_idx'][i])
            preprocessed_dates.append(batch['encoded_input'][i])
            coords.append(batch['coords'][i])
        except Exception as e:
            failed += 1
            print(f"Skipping {url}: {type(e).__name__}: {e}")
            continue

    if len(preprocessed_images) >= target:
        break

print(f"Collected {len(preprocessed_images)}, failed {failed}")
assert len(preprocessed_images) == len(test_labels) == len(preprocessed_dates) == len(test_images)

Skipping https://inaturalist-open-data.s3.amazonaws.com/photos/56922419/original.jpg: UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x000001E6655A9580>
Skipping https://inaturalist-open-data.s3.amazonaws.com/photos/354065334/original.jpeg: UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x000001E660DC2200>
Collected 1000, failed 2


baseline CLIP zero-shot performance on the test set

In [19]:

prob_list = []
correct_predictions = 0
for i, img in enumerate(preprocessed_images):
    img = img.to(device)
    with torch.no_grad(), torch.amp.autocast("cuda"):
        image_features = model.encode_image(img)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_probs = (100.0 * image_features @ species_text_embs.T.to(device)).softmax(dim=-1)
        predicted_idx = text_probs.argmax().item()
        if predicted_idx == test_labels[i]:
            correct_predictions += 1
        print(f"Test Image {i}: Predicted species: {unique_names[predicted_idx]}, Prob: {text_probs[0, predicted_idx].item():.4f}, Correct: {predicted_idx == test_labels[i]}")
        prob_list.append(text_probs[0, predicted_idx].item())

print(f"Average predicted probability for the correct species across test images: {np.mean(prob_list):.4f}")
print(f"Correct predictions: {correct_predictions}/{len(test_labels)}")

Test Image 0: Predicted species: Sturnus vulgaris, Prob: 0.9986, Correct: True
Test Image 1: Predicted species: Leucanthemum vulgare, Prob: 0.8869, Correct: True
Test Image 2: Predicted species: Amanita muscaria, Prob: 0.8546, Correct: True
Test Image 3: Predicted species: Lissachatina fulica, Prob: 0.4179, Correct: False
Test Image 4: Predicted species: Zonotrichia albicollis, Prob: 0.9955, Correct: True
Test Image 5: Predicted species: Leucanthemum vulgare, Prob: 0.6467, Correct: True
Test Image 6: Predicted species: Steatoda nobilis, Prob: 0.9972, Correct: True
Test Image 7: Predicted species: Podarcis muralis, Prob: 0.7845, Correct: False
Test Image 8: Predicted species: Cepaea nemoralis, Prob: 0.9996, Correct: True
Test Image 9: Predicted species: Haemorhous mexicanus, Prob: 0.8158, Correct: True
Test Image 10: Predicted species: Junco hyemalis, Prob: 0.9942, Correct: True
Test Image 11: Predicted species: Pholcus phalangioides, Prob: 1.0000, Correct: True
Test Image 12: Predicted

In [20]:
# Precompute once, outside the loop
label_array = np.array(processed['label_idx'], dtype=np.int64)
counts = torch.bincount(torch.from_numpy(label_array), minlength=len(name_to_idx)).float()
log_prior = torch.log(counts / counts.sum() + 1e-12).to(device)

**The idea is now that if I take image metadata (such as date), I could better defirentiate the species**

In [ ]:
from torch.nn import functional as F


network_date_to_species.eval()
species_text_embs_gpu = species_text_embs.to(device)   


clip_correct = 0
fused_correct = 0
fused_confidences = []
total = 0

for i, img in enumerate(preprocessed_images):
    img = img.to(device)
    true_label = test_labels[i]   

    with torch.no_grad(), torch.amp.autocast("cuda"):
        #  CLIP branch 
        image_features = model.encode_image(img)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        clip_logits = 100.0 * image_features @ species_text_embs_gpu.T     
        clip_log_probs = F.log_softmax(clip_logits.float(), dim=-1)        # normalize between models

        #  Date branch 
        date_input = preprocessed_dates[i].unsqueeze(0).to(device)
        date_logits = network_date_to_species(date_input)                   
        date_log_probs = F.log_softmax(date_logits, dim=-1)                

        #  Bayesian fusion in log-space 
        fused_log_probs = clip_log_probs + date_log_probs - log_prior
        fused_log_probs = F.log_softmax(fused_log_probs, dim=-1)           

        clip_pred  = clip_log_probs.argmax(dim=-1).item()
        fused_pred = fused_log_probs.argmax(dim=-1).item()
        fused_prob = fused_log_probs[0, fused_pred].exp().item()

    clip_correct  += int(clip_pred  == true_label)
    fused_correct += int(fused_pred == true_label)
    fused_confidences.append(fused_prob)
    total += 1

    print(f"Image {i}: CLIP={unique_names[clip_pred]}, "
          f"Fused={unique_names[fused_pred]} (p={fused_prob:.3f}), "
          f"True={unique_names[true_label]}")

print(f"\nCLIP-only accuracy:  {clip_correct/total:.3f}")
print(f"Fused accuracy:      {fused_correct/total:.3f}")
print(f"Avg fused confidence: {np.mean(fused_confidences):.4f}")

Image 0: CLIP=Sturnus vulgaris, Fused=Sturnus vulgaris (p=0.999), True=Sturnus vulgaris
Image 1: CLIP=Leucanthemum vulgare, Fused=Leucanthemum vulgare (p=0.939), True=Leucanthemum vulgare
Image 2: CLIP=Amanita muscaria, Fused=Amanita muscaria (p=0.845), True=Amanita muscaria
Image 3: CLIP=Lissachatina fulica, Fused=Lissachatina fulica (p=0.517), True=Hygromia cinctella
Image 4: CLIP=Zonotrichia albicollis, Fused=Zonotrichia albicollis (p=0.995), True=Zonotrichia albicollis
Image 5: CLIP=Leucanthemum vulgare, Fused=Leucanthemum vulgare (p=0.804), True=Leucanthemum vulgare
Image 6: CLIP=Steatoda nobilis, Fused=Steatoda nobilis (p=0.998), True=Steatoda nobilis
Image 7: CLIP=Podarcis muralis, Fused=Podarcis muralis (p=0.796), True=Zootoca vivipara
Image 8: CLIP=Cepaea nemoralis, Fused=Cepaea nemoralis (p=1.000), True=Cepaea nemoralis
Image 9: CLIP=Haemorhous mexicanus, Fused=Haemorhous mexicanus (p=0.846), True=Haemorhous mexicanus
Image 10: CLIP=Junco hyemalis, Fused=Junco hyemalis (p=0.9

# Experiment 2? Same Idea but with location of the image! 

In [31]:
import torch.nn as nn

# This network is prett naive and prob can be improved

class Loc_to_species_Common(nn.Module):
    """Maps 2-dim location features into one of Cryptic Bio species Scientific Name"""
    def __init__(self, output_dim=158): # 158 unique species in the common subset
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim),
                
        )

    def forward(self, x):
       x=torch.flatten(x, start_dim=1)  # Flatten the input to (batch_size, coord_dim)
       return self.network(x)
        

No need to encode coordinates, just feed them in as is to the network!

In [32]:
network_coords_to_species = Loc_to_species_Common().to(device)
loss=nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(network_coords_to_species.parameters(), lr=1e-3)
for epoch in range(10):
    total_loss, steps = 0.0, 0
    for batch in train_loader:
        x = batch['coords'].to(device, non_blocking=True)
        y = batch['label_idx'].to(device, non_blocking=True)
        optimizer.zero_grad()
        logits = network_coords_to_species(x)
        l = loss(logits, y)
        l.backward()
        optimizer.step()
        total_loss += l.item(); steps += 1
    print(f"Epoch {epoch} | train loss: {total_loss/steps:.4f}")

Epoch 0 | train loss: 3.3171
Epoch 1 | train loss: 3.1758
Epoch 2 | train loss: 3.1423
Epoch 3 | train loss: 3.1234
Epoch 4 | train loss: 3.1097
Epoch 5 | train loss: 3.1005
Epoch 6 | train loss: 3.0934
Epoch 7 | train loss: 3.0869
Epoch 8 | train loss: 3.0828
Epoch 9 | train loss: 3.0783


In [ ]:
from torch.nn import functional as F


network_coords_to_species.eval()
species_text_embs_gpu = species_text_embs.to(device)   

clip_correct_2 = 0
fused_correct_2 = 0
fused_confidences_2 = []
total = 0

for i, img in enumerate(preprocessed_images):
    img = img.to(device)
    true_label = test_labels[i] 

    with torch.no_grad(), torch.amp.autocast("cuda"):
        #  CLIP branch 
        image_features = model.encode_image(img)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        clip_logits = 100.0 * image_features @ species_text_embs_gpu.T     
        clip_log_probs = F.log_softmax(clip_logits.float(), dim=-1)        # normalize between models

        #  Coordinates branch 
        coord_input = coords[i].unsqueeze(0).to(device)
        coord_logits = network_coords_to_species(coord_input)                   
        coord_log_probs = F.log_softmax(coord_logits, dim=-1)                

        #  Bayesian fusion in log-space 
        fused_log_probs = clip_log_probs + coord_log_probs - log_prior
        fused_log_probs = F.log_softmax(fused_log_probs, dim=-1)           

        clip_pred  = clip_log_probs.argmax(dim=-1).item()
        fused_pred = fused_log_probs.argmax(dim=-1).item()
        fused_prob = fused_log_probs[0, fused_pred].exp().item()

    clip_correct_2  += int(clip_pred  == true_label)
    fused_correct_2 += int(fused_pred == true_label)
    fused_confidences_2.append(fused_prob)
    total += 1

    print(f"Image {i}: CLIP={unique_names[clip_pred]}, "
          f"Fused={unique_names[fused_pred]} (p={fused_prob:.3f}), "
          f"True={unique_names[true_label]}")

print(f"\nCLIP-only accuracy:  {clip_correct_2/total:.3f}")
print(f"Fused accuracy:      {fused_correct_2/total:.3f}")
print(f"Avg fused confidence: {np.mean(fused_confidences_2):.4f}")

Image 0: CLIP=Sturnus vulgaris, Fused=Sturnus vulgaris (p=0.997), True=Sturnus vulgaris
Image 1: CLIP=Leucanthemum vulgare, Fused=Leucanthemum vulgare (p=0.924), True=Leucanthemum vulgare
Image 2: CLIP=Amanita muscaria, Fused=Amanita muscaria (p=0.943), True=Amanita muscaria
Image 3: CLIP=Lissachatina fulica, Fused=Cepaea nemoralis (p=0.444), True=Hygromia cinctella
Image 4: CLIP=Zonotrichia albicollis, Fused=Zonotrichia albicollis (p=0.999), True=Zonotrichia albicollis
Image 5: CLIP=Leucanthemum vulgare, Fused=Leucanthemum vulgare (p=0.767), True=Leucanthemum vulgare
Image 6: CLIP=Steatoda nobilis, Fused=Steatoda nobilis (p=1.000), True=Steatoda nobilis
Image 7: CLIP=Podarcis muralis, Fused=Podarcis muralis (p=0.999), True=Zootoca vivipara
Image 8: CLIP=Cepaea nemoralis, Fused=Cepaea nemoralis (p=1.000), True=Cepaea nemoralis
Image 9: CLIP=Haemorhous mexicanus, Fused=Haemorhous mexicanus (p=0.836), True=Haemorhous mexicanus
Image 10: CLIP=Junco hyemalis, Fused=Junco hyemalis (p=0.998)